# Decentralized world model

> One world model per agent, each seeing only its own observations and planning its own actions.

In [ ]:
#| default_exp wm.decentralized

In [ ]:
#| hide
from nbdev.showdoc import *

`DecentralizedWorldModel` holds **one world model per agent, with its own weights**. Agent ``a``'s
model only ever reads agent ``a``'s slice of the infos: its own observation frames (by default its
RGB ``pov``), its own executed actions (``action_history``), its own goal image, and its own
candidate actions. Its predictions therefore depend on nothing of the other agents.

It implements the `Dynamics` protocol (``encode`` / ``rollout`` with an agent axis), so it plugs into
`ShootingCostEvaluator`. With a per-agent `GoalMSE` and a solver in ``mode='per_agent'``, each
agent's cost depends only on its own actions through its own model: every agent plans
independently, with its own world model, from its own local view.

Models with extra inputs (e.g. ``extra_inputs={'position': 2}``) also read their agent's slice of
those infos. Discrete actions are fed to the models one-hot (the form of the categorical solvers' candidates);
executed actions (dataset, ``action_history``) are converted from action indices.

In [ ]:
#| export
from __future__ import annotations

import copy
from pathlib import Path
from typing import Any

import torch
from torch import nn

from stable_marl.wm.lewm.lewm import LeWM, build_lewm

In [ ]:
#| export
class DecentralizedWorldModel(nn.Module):
    """
    Parameters
    ----------
    models : list of nn.Module
        One world model per agent (e.g. :class:`LeWM`), each with ``encode`` / ``rollout``
    obs_key : str
        Info key of the agents' observation frames, ``(B, T, num_agents, ...)``
    num_actions : int, optional
        Number of discrete actions (actions fed one-hot); None for continuous actions
    action_block : int
        Env steps per planning step
    """
    def __init__(self, models: list[nn.Module], obs_key: str = 'pov', num_actions: int | None = None,
                 action_block: int = 1, config: dict | None = None):
        super().__init__()
        self.models = nn.ModuleList(models)
        self.obs_key, self.num_actions, self.action_block = obs_key, num_actions, action_block
        self.config = config   # how the models were built (for save / load)

    @classmethod
    def build_lewm(cls, num_agents: int, num_actions: int | None = None, action_size: int | None = None,
                   action_block: int = 1, obs_key: str = 'pov', **lewm_kwargs) -> DecentralizedWorldModel:
        """
        One independent :func:`build_lewm` per agent, for discrete actions (`num_actions`) or continuous
        actions of size `action_size`.
        """
        if (num_actions is None) == (action_size is None):
            raise ValueError("Pass exactly one of num_actions (discrete) or action_size (continuous)")
        action_dim = action_block * (num_actions or action_size)
        models = [build_lewm(action_dim=action_dim, obs_key='pixels', **lewm_kwargs) for _ in range(num_agents)]
        config = dict(num_agents=num_agents, num_actions=num_actions, action_size=action_size,
                      action_block=action_block, obs_key=obs_key, **lewm_kwargs)
        return cls(models, obs_key=obs_key, num_actions=num_actions, action_block=action_block, config=config)

    @property
    def num_agents(self) -> int:
        return len(self.models)

    def agent_actions(self, actions: torch.Tensor) -> torch.Tensor:
        """
        Executed actions of one agent, ``(..., action_block)`` indices (discrete) or
        ``(..., action_block * D)`` values -> model input ``(..., action_block * K)`` one-hot / values.
        NaNs (no action, e.g. after a reset) become zeros.
        """
        actions = torch.nan_to_num(actions.float(), nan=-1.0 if self.num_actions else 0.0)
        if not self.num_actions:
            return actions
        one_hot = torch.nn.functional.one_hot(actions.long().clamp(min=0), self.num_actions).float()
        return (one_hot * (actions >= 0).unsqueeze(-1)).flatten(-2)

    def _agent_inputs(self, x: dict, model: nn.Module, a: int, agent_axis: int) -> dict:
        "Agent `a`'s slice of the inputs `model` reads (frames under the model's own key, extra inputs)."
        inputs = {model.obs_key: x[self.obs_key].select(agent_axis, a)}
        for key in getattr(model, 'extra_encoders', {}):
            inputs[key] = x[key].select(agent_axis, a)
        return inputs

    def encode(self, x: dict) -> dict:
        "Adds ``emb`` ``(B, T, num_agents, D)``: each agent's inputs encoded by its own model."
        embs = [model.encode(self._agent_inputs(x, model, a, 2))['emb'] for a, model in enumerate(self.models)]
        x['emb'] = torch.stack(embs, dim=2)
        return x

    def rollout(self, info_dict: dict, action_candidates: torch.Tensor) -> dict:
        """
        Roll each agent's candidates ``(B, S, horizon, num_agents, action_dim)`` out with its own
        model, from its own context. Adds ``predicted_emb`` ``(B, S, H + horizon, num_agents, D)``
        (and caches the encoded context ``emb``).
        """
        history = info_dict.get('action_history')                # (B, S, H - 1, A, action_block * D)
        cached = info_dict.get('emb')
        preds, embs = [], []
        for a, model in enumerate(self.models):
            agent = self._agent_inputs(info_dict, model, a, 3)
            if history is not None:
                agent['action_history'] = self.agent_actions(history[:, :, :, a])
            if cached is not None:
                agent['emb'] = cached[:, :, :, a]
            agent = model.rollout(agent, action_candidates[:, :, :, a])
            preds.append(agent['predicted_emb'])
            embs.append(agent['emb'])
        info_dict['emb'] = torch.stack(embs, dim=3)
        info_dict['predicted_emb'] = torch.stack(preds, dim=3)
        return info_dict

    def save(self, path: str | Path):
        "Save the weights and the build configuration (models made with :meth:`build_lewm`)."
        torch.save({'config': self.config, 'state_dict': self.state_dict()}, path)

    @classmethod
    def load(cls, path: str | Path, map_location: Any = 'cpu') -> DecentralizedWorldModel:
        checkpoint = torch.load(path, map_location=map_location, weights_only=False)
        model = cls.build_lewm(**checkpoint['config'])
        model.load_state_dict(checkpoint['state_dict'])
        return model

### Tests

In [ ]:
import tempfile
from stable_marl.protocols import Dynamics

TINY = dict(image_size=32, patch_size=8, embed_dim=32, depth=2, heads=2, dim_head=16, mlp_dim=64, projector_hidden=64,
            encoder_kwargs=dict(dim=32, depth=2, heads=2, mlp_dim=64), history_size=3)
wm = DecentralizedWorldModel.build_lewm(num_agents=2, num_actions=4, action_block=2, **TINY).eval()
assert isinstance(wm, Dynamics) and wm.num_agents == 2
# separate weights per agent
assert not torch.equal(wm.models[0].encoder.pos_embedding, wm.models[1].encoder.pos_embedding)

B, S, H, T = 2, 3, 2, 4
info = {'pov': torch.randint(0, 256, (B, S, H, 2, 56, 56, 3), dtype=torch.uint8),
        'action_history': torch.randint(0, 4, (B, S, H - 1, 2, 2)).float()}
cand = torch.nn.functional.one_hot(torch.randint(0, 4, (B, S, T, 2, 2)), 4).float().flatten(-2)  # (B, S, T, A, 2*4)
out = wm.rollout(dict(info), cand)
assert out['predicted_emb'].shape == (B, S, H + T, 2, 32)

# decentralized: agent 0's predictions ignore everything of agent 1
other = dict(info); other['pov'] = info['pov'].clone(); other['pov'][:, :, :, 1] = 0
other['action_history'] = info['action_history'].clone(); other['action_history'][..., 1, :] = 3
cand2 = cand.clone(); cand2[:, :, :, 1] = cand.flip(0)[:, :, :, 1]
out2 = wm.rollout(other, cand2)
assert torch.allclose(out2['predicted_emb'][..., 0, :], out['predicted_emb'][..., 0, :], atol=1e-5)
assert not torch.allclose(out2['predicted_emb'][..., 1, :], out['predicted_emb'][..., 1, :])

assert wm.agent_actions(torch.tensor([[2.0, float('nan')]])).tolist() == [[0, 0, 1, 0, 0, 0, 0, 0]]
with tempfile.TemporaryDirectory() as tmp:
    wm.save(f'{tmp}/wm.pt')
    loaded = DecentralizedWorldModel.load(f'{tmp}/wm.pt').eval()
    assert torch.allclose(loaded.rollout(dict(info), cand)['predicted_emb'], out['predicted_emb'], atol=1e-6)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()